# Progetti - by Torlone

Si consideri lo schema relazionale composto dalle seguenti relazioni:

- Impiegato (**Matricola**,Cognome,Stipendio,*Dipartimento*)
- Dipartimento (**Codice**,Nome,Sede,*Direttore*)
- Progetto (**Sigla**,Nome,Bilancio,*Responsabile*)
- Partecipazione (_**Impiegato**_,_**Progetto**_)

con i seguenti vincoli di riferimento:

- tra l'attributo Dipartimento della relazione Impiegato e la relazione Dipartimento
- tra l'attributo Direttore della relazione Dipartimento e la relazione Impiegato
- tra l'attributo Responsabile della relazione Progetto e la relazione Impiegato
- tra l'attributo Impiegato della relazione Partecipazione e la relazione Impiegato
- tra l'attributo Progetto della relazione Partecipazione e la relazione Progetto

```txt
     ┌─────────────────────┐ 
     │Dipartimento         │ 
     ├─────────────────────┤ 
     │**codice**: number   │ 
     │nome: string         │ 
     │sede: string         │ 
     │//direttore//: number│ 
     └─────────────────────┘ 
               |             
               |             
   ┌────────────────────────┐
   │Impiegato               │
   ├────────────────────────┤
   │**matricola**: number   │
   │cognome: string         │
   │stipendio: number       │
   │//dipartimento//: number│
   └────────────────────────┘
                    |         
┌─────────────────┐ |        
│Partecipazione   │ |        
├─────────────────┤ |        
│**//impiegato//**│-|        
│**//progetto//** │ |        
└─────────────────┘ |        
                    |        
                    |         
   ┌────────────────────────┐
   │Progetto                │
   ├────────────────────────┤
   │**sigla**: string       │
   │nome: string            │
   │bilancio: number        │
   │//responsabile//: number│
   └────────────────────────┘
```

Formulare le seguenti interrogazioni in algebra relazionale e SQL.

1. Trovare matricola e cognome degli impiegati che guadagnano più di 50 mila euro.
2. Trovare cognome e stipendio degli impiegati che lavorano a Roma.
3. Trovare cognome degli impiegati e nome del dipartimento in cui lavorano.
4. Trovare cognome degli impiegati che sono direttori di dipartimento.
5. Trovare i nomi dei progetti e i cognomi dei responsabili.
6. Trovare i nomi dei progetti con bilancio maggiore di 100.000 e i cognomi degli impiegati che lavorano su di essi.
7. Trovare il cognome degli impiegati che guadagnano più del loro direttore di dipartimento.
8. Trovare cognome dei direttori di dipartimento e dei responsabili di progetto.
9. Trovare nomi dei dipartimenti in cui lavorano impiegati che guadagnano più di 60K.
10. Trovare nomi dei dipartimenti in cui tutti gli impiegati guadagnano più di 60K.
11. Trovare cognome degli impiegati di stipendio massimo.
12. Trovare matricola e cognome degli impiegati che non lavorano a nessun progetto.
13. Trovare matricola e cognome degli impiegati che lavorano a più di un progetto.
14. Trovare matricola e cognome degli impiegati che lavorano a un solo progetto.

In [1]:
import urllib.parse
from IPython.display import Image, display
import base64
import zlib

def draw_uml(uml_code):
    # Rimuove spazi vuoti e codifica per URL
    encoded = urllib.parse.quote(uml_code)
    url = base64.urlsafe_b64encode(zlib.compress(uml_code.encode('utf-8'), 9)).decode('ascii')
    url = f"https://kroki.io/plantuml/svg/{url}"
    # print(url)
    display({'text/html': f'<img src="{url}">'}, raw=True)

diagramma = """
@startuml
hide circle
hide methods
left to right direction 

class Dipartimento{
  **codice**: number
  nome: string
  sede: string
  //direttore//: number
}

class Impiegato {
	**matricola**: number
	cognome: string
	stipendio: number
	//dipartimento//: number
}

Impiegato "1" -- "0..1" Dipartimento : diretto <
Dipartimento "1" -- "n" Impiegato : afferisce <

class Partecipazione {
  **//impiegato//**
  **//progetto//**
}

class Progetto {
  **sigla**: string
  nome: string
  bilancio: number
  //responsabile//: number
}

Impiegato "1" -- "n" Partecipazione
Partecipazione "n" -- "1" Progetto

Impiegato "1" -- "0..n" Progetto : responsabile <
@enduml
"""

draw_uml(diagramma)

In [2]:
import sqlite3
import pandas as pd

# Connessione a un database in memoria (non si corrompe mai)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

sql_script = """
PRAGMA foreign_keys = OFF;

-- Rimozione tabelle in ordine inverso di dipendenza
DROP TABLE IF EXISTS Partecipazione;
DROP TABLE IF EXISTS Progetto;
DROP TABLE IF EXISTS Impiegato;
DROP TABLE IF EXISTS Dipartimento;

CREATE TABLE Impiegato (
    matricola INTEGER PRIMARY KEY,
    cognome TEXT NOT NULL,
    stipendio REAL CHECK (stipendio >= 0),
    dipartimento INTEGER,
    CONSTRAINT fk_afferisce FOREIGN KEY (dipartimento) REFERENCES Dipartimento(codice) ON DELETE SET NULL
);

CREATE TABLE Dipartimento (
    codice INTEGER PRIMARY KEY,
    nome TEXT NOT NULL,
    sede TEXT,
    direttore INTEGER,
    CONSTRAINT fk_diretto FOREIGN KEY (direttore) REFERENCES Impiegato(matricola) ON DELETE SET NULL
);

CREATE TABLE Progetto (
    sigla TEXT PRIMARY KEY,
    nome TEXT NOT NULL,
    bilancio REAL,
    responsabile INTEGER,
    CONSTRAINT fk_responsabile FOREIGN KEY (responsabile) REFERENCES Impiegato(matricola) ON DELETE CASCADE
);

CREATE TABLE Partecipazione (
    impiegato INTEGER,
    progetto TEXT,
    PRIMARY KEY (impiegato, progetto), -- Chiave composta
    FOREIGN KEY (impiegato) REFERENCES Impiegato(matricola) ON DELETE CASCADE,
    FOREIGN KEY (progetto) REFERENCES Progetto(sigla) ON DELETE CASCADE
);

INSERT INTO Impiegato VALUES 
    (1, 'Rossi', 63000, 1),
	(2, 'Verdi', 58000, 2),
	(3, 'Bianchi', 77000, 3),
	(4, 'Marroni', 51000, 4),
	(5, 'Gialli', 19000, 1),
	(6, 'Viola', 42000, 5),
	(7, 'Arancioni', 35000, 3),
	(8, 'Celesti', 36000, 3),
	(9, 'Ciano', 24500, 3),
	(10, 'Panna', 65000, 4),
	(11, 'Rosa', 49000, 5);

INSERT INTO Dipartimento VALUES
	(1, 'Amministrazione', 'Roma', 1),
	(2, 'Produzione', 'Ancona', 2),
	(3, 'Marketing', 'Roma', 3),
	(4, 'Vendite', 'Milano', 4),
	(5, 'Risorse umane', 'Milano', 6);

INSERT INTO Progetto VALUES
  ('NCP', 'Nuova campagna pubblicitaria', 50000, 3),
  ('RCD', 'Riorganizzazione canali digitali', 15000, 3),
  ('SEL', 'Nuovo modulo vendite ERP', 250000, 4),
  ('FID', 'Integrazione programma fedeltà nuovo modulo vendite', 80000, 4),
  ('OTT', 'Ottimizzazione del processo produttivo', 300000, 2),
  ('CTR', 'Controllo del processo produttivo e analisi tempi', 10000, 2),
  ('EDU', 'Piano di formazione del personale neoassunto', 10000, 5);

INSERT INTO Partecipazione VALUES
  (2, 'OTT'),
  (2, 'CTR'),
  (4, 'OTT'),
  (6, 'CTR'),
  (7, 'NCP'),
  (7, 'RCD'),
  (8, 'NCP'),
  (8, 'RCD'),
  (9, 'NCP'),
  (10, 'FID'),
  (10, 'SEL'),
  (11, 'EDU');
  
PRAGMA foreign_key_check;
"""

# Esecuzione
cursor.executescript(sql_script)

def q(query):
    return pd.read_sql_query(query, conn)

tables = ["Impiegato", "Dipartimento", "Progetto", "Partecipazione"]
for t in tables:
    print(f"\n--- Tabella: {t} ---")
    display(q(f"SELECT * FROM {t}"))


--- Tabella: Impiegato ---


,matricola,cognome,stipendio,dipartimento
0,1,Rossi,63000.0,1
1,2,Verdi,58000.0,2
2,3,Bianchi,77000.0,3
3,4,Marroni,51000.0,4
4,5,Gialli,19000.0,1
5,6,Viola,42000.0,5
6,7,Arancioni,35000.0,3
7,8,Celesti,36000.0,3
8,9,Ciano,24500.0,3
9,10,Panna,65000.0,4



--- Tabella: Dipartimento ---


,codice,nome,sede,direttore
0,1,Amministrazione,Roma,1
1,2,Produzione,Ancona,2
2,3,Marketing,Roma,3
3,4,Vendite,Milano,4
4,5,Risorse umane,Milano,6



--- Tabella: Progetto ---


,sigla,nome,bilancio,responsabile
0,NCP,Nuova campagna pubblicitaria,50000.0,3
1,RCD,Riorganizzazione canali digitali,15000.0,3
2,SEL,Nuovo modulo vendite ERP,250000.0,4
3,FID,Integrazione programma fedeltà nuovo modulo ve...,80000.0,4
4,OTT,Ottimizzazione del processo produttivo,300000.0,2
5,CTR,Controllo del processo produttivo e analisi tempi,10000.0,2
6,EDU,Piano di formazione del personale neoassunto,10000.0,5



--- Tabella: Partecipazione ---


,impiegato,progetto
0,2,OTT
1,2,CTR
2,4,OTT
3,6,CTR
4,7,NCP
5,7,RCD
6,8,NCP
7,8,RCD
8,9,NCP
9,10,FID


In [3]:
# 1. Trovare matricola e cognome degli impiegati che guadagnano più di 50 mila euro.
print("""
π matricola, cognome (
	σ stipendio ≥ 50000 (
		Impiegato
	)
)
""")

q("""
SELECT
      matricola
    , cognome
FROM
    Impiegato
WHERE
    stipendio >= 50000;
""")

,matricola,cognome
0,1,Rossi
1,2,Verdi
2,3,Bianchi
3,4,Marroni
4,10,Panna


In [4]:
# 2. Trovare cognome e stipendio degli impiegati che lavorano a Roma.
print("""
π cognome, stipendio (
		σ sede='Roma' (
			Impiegato ⨝ dipartimento=codice Dipartimento
		)
)
""")

q("""
SELECT
      cognome
    , stipendio
FROM
    Impiegato INNER JOIN
    Dipartimento
        ON Impiegato.dipartimento = Dipartimento.codice
WHERE
    sede = 'Roma';
""")

,cognome,stipendio
0,Rossi,63000.0
1,Bianchi,77000.0
2,Gialli,19000.0
3,Arancioni,35000.0
4,Celesti,36000.0
5,Ciano,24500.0


In [6]:
# 3. Trovare cognome degli impiegati e nome del dipartimento in cui lavorano.
print("""
π cognome, nome (
		Impiegato ⨝ dipartimento=codice Dipartimento
)
""")

q("""
SELECT
      cognome
    , nome
FROM
    Impiegato INNER JOIN
    Dipartimento
        ON Impiegato.dipartimento = Dipartimento.codice
""")


π cognome, nome (
		Impiegato ⨝ dipartimento=codice Dipartimento
)



,cognome,nome
0,Rossi,Amministrazione
1,Verdi,Produzione
2,Bianchi,Marketing
3,Marroni,Vendite
4,Gialli,Amministrazione
5,Viola,Risorse umane
6,Arancioni,Marketing
7,Celesti,Marketing
8,Ciano,Marketing
9,Panna,Vendite


In [7]:
# 4. Trovare cognome degli impiegati che sono direttori di dipartimento.
print("""
π cognome (
    Impiegato ⨝ matricola=direttore Dipartimento
)
""")

q("""
SELECT
    cognome
FROM
    Impiegato INNER JOIN
    Dipartimento ON matricola = direttore;
""")


π cognome (
    Impiegato ⨝ matricola=direttore Dipartimento
)



,cognome
0,Rossi
1,Verdi
2,Bianchi
3,Marroni
4,Viola


In [8]:
# 5. Trovare i nomi dei progetti e i cognomi dei responsabili.
print("""
π nome, cognome (
    Progetto ⨝ responsabile=matricola Impiegato
)
""")

q("""
SELECT
      nome
    , cognome
FROM
    Progetto INNER JOIN
    Impiegato ON responsabile = matricola;
""")


π nome, cognome (
    Progetto ⨝ responsabile=matricola Impiegato
)



,nome,cognome
0,Nuova campagna pubblicitaria,Bianchi
1,Riorganizzazione canali digitali,Bianchi
2,Nuovo modulo vendite ERP,Marroni
3,Integrazione programma fedeltà nuovo modulo ve...,Marroni
4,Ottimizzazione del processo produttivo,Verdi
5,Controllo del processo produttivo e analisi tempi,Verdi
6,Piano di formazione del personale neoassunto,Gialli


In [10]:
# 6. Trovare i nomi dei progetti con bilancio maggiore di 100.000 e i cognomi degli impiegati che lavorano su di essi.
print("""
π nome, cognome (
    σ bilancio > 100000 (Progetto) ⨝ sigla=progetto Partecipazione ⨝ impiegato=matricola Impiegato
)
""")

q("""
SELECT P.nome, I.cognome
FROM Progetto P
JOIN Partecipazione Part ON P.sigla = Part.progetto
JOIN Impiegato I ON Part.impiegato = I.matricola
WHERE P.bilancio > 100000;
""")

# 7. Trovare il cognome degli impiegati che guadagnano più del loro direttore di dipartimento.
print("""
π I1.cognome (
    (Impiegato AS I1 ⨝ I1.dipartimento=D.codice Dipartimento AS D) 
    ⨝ D.direttore=I2.matricola (Impiegato AS I2)
    σ I1.stipendio > I2.stipendio
)
""")

q("""
SELECT I1.cognome
FROM Impiegato I1
JOIN Dipartimento D ON I1.dipartimento = D.codice
JOIN Impiegato I2 ON D.direttore = I2.matricola
WHERE I1.stipendio > I2.stipendio;
""")

# 8. Trovare cognome dei direttori di dipartimento e dei responsabili di progetto.
print("""
π cognome (Impiegato ⨝ matricola=direttore Dipartimento) 
∪ 
π cognome (Impiegato ⨝ matricola=responsabile Progetto)
""")

q("""
SELECT cognome FROM Impiegato JOIN Dipartimento ON matricola = direttore
UNION
SELECT cognome FROM Impiegato JOIN Progetto ON matricola = responsabile;
""")

# 9. Trovare nomi dei dipartimenti in cui lavorano impiegati che guadagnano più di 60K.
print("""
π nome (Dipartimento ⨝ codice=dipartimento σ stipendio > 60000 (Impiegato))
""")

q("""
SELECT DISTINCT D.nome
FROM Dipartimento D
JOIN Impiegato I ON D.codice = I.dipartimento
WHERE I.stipendio > 60000;
""")

# 10. Trovare nomi dei dipartimenti in cui tutti gli impiegati guadagnano più di 60K.
# (Sottrazione: Tutti i dipartimenti - Dipartimenti che hanno almeno un impiegato che guadagna <= 60K)
print("""
π nome (Dipartimento) - π nome (Dipartimento ⨝ codice=dipartimento σ stipendio ≤ 60000 (Impiegato))
""")

q("""
SELECT nome FROM Dipartimento
EXCEPT
SELECT D.nome FROM Dipartimento D 
JOIN Impiegato I ON D.codice = I.dipartimento 
WHERE I.stipendio <= 60000;
""")

# 11. Trovare cognome degli impiegati di stipendio massimo.
print("""
π cognome (Impiegato) - π I1.cognome (Impiegato AS I1 ⨝ I1.stipendio < I2.stipendio Impiegato AS I2)
""")

q("""
SELECT cognome 
FROM Impiegato 
WHERE stipendio = (SELECT MAX(stipendio) FROM Impiegato);
""")

# 12. Trovare matricola e cognome degli impiegati che non lavorano a nessun progetto.
print("""
π matricola, cognome (Impiegato) - π matricola, cognome (Impiegato ⨝ matricola=impiegato Partecipazione)
""")

q("""
SELECT matricola, cognome
FROM Impiegato
WHERE matricola NOT IN (SELECT impiegato FROM Partecipazione);
""")

# 13. Trovare matricola e cognome degli impiegati che lavorano a più di un progetto.
# In algebra relazionale si usa il join di Partecipazione con se stessa su sigle progetto diverse
print("""
π matricola, cognome (Impiegato ⨝ matricola=P1.impiegato (Partecipazione AS P1 ⨝ P1.impiegato=P2.impiegato ∧ P1.progetto≠P2.progetto Partecipazione AS P2))
""")

q("""
SELECT I.matricola, I.cognome
FROM Impiegato I
JOIN Partecipazione P ON I.matricola = P.impiegato
GROUP BY I.matricola, I.cognome
HAVING COUNT(P.progetto) > 1;
""")

# 14. Trovare matricola e cognome degli impiegati che lavorano a un solo progetto.
q("""
SELECT I.matricola, I.cognome
FROM Impiegato I
JOIN Partecipazione P ON I.matricola = P.impiegato
GROUP BY I.matricola, I.cognome
HAVING COUNT(P.progetto) = 1;
""")


π nome, cognome (
    σ bilancio > 100000 (Progetto) ⨝ sigla=progetto Partecipazione ⨝ impiegato=matricola Impiegato
)


π I1.cognome (
    (Impiegato AS I1 ⨝ I1.dipartimento=D.codice Dipartimento AS D) 
    ⨝ D.direttore=I2.matricola (Impiegato AS I2)
    σ I1.stipendio > I2.stipendio
)


π cognome (Impiegato ⨝ matricola=direttore Dipartimento) 
∪ 
π cognome (Impiegato ⨝ matricola=responsabile Progetto)


π nome (Dipartimento ⨝ codice=dipartimento σ stipendio > 60000 (Impiegato))


π nome (Dipartimento) - π nome (Dipartimento ⨝ codice=dipartimento σ stipendio ≤ 60000 (Impiegato))


π cognome (Impiegato) - π I1.cognome (Impiegato AS I1 ⨝ I1.stipendio < I2.stipendio Impiegato AS I2)


π matricola, cognome (Impiegato) - π matricola, cognome (Impiegato ⨝ matricola=impiegato Partecipazione)


π matricola, cognome (Impiegato ⨝ matricola=P1.impiegato (Partecipazione AS P1 ⨝ P1.impiegato=P2.impiegato ∧ P1.progetto≠P2.progetto Partecipazione AS P2))



,matricola,cognome
0,4,Marroni
1,6,Viola
2,9,Ciano
3,11,Rosa
